# 013 — Espacios de estados y formulación de problemas

Este notebook reutiliza el mismo núcleo ejecutable que `lab.py`. El objetivo no es
ocultar la implementación, sino separar exploración, ejercicio y solución.

**Evidencia esperada:** resultado JSON, interpretación de una decisión y una
limitación documentada.


In [ ]:
from ai_evolution.labs import run_lab
import json

def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


## Solución de referencia

La solución valida el contrato mínimo sin asumir un valor interno específico.


In [ ]:
result = run_lab("search", seed=13)
assert result["kind"] == "search"
assert result["evidence"]
show(result)


## Solución 1 — Jarras 5 L / 2 L, objetivo 1 L

a) **Estado**: par `(x, y)` con `0 ≤ x ≤ 5`, `0 ≤ y ≤ 2`. `s0 = (0, 0)`.
**Acciones**: llenar5, llenar2, vaciar5, vaciar2, verter5→2, verter2→5.
**IS-GOAL**: `x = 1`. **Costo**: 1 por acción.

b) `6 × 3 = 18` estados posibles: el estado es el producto cartesiano de los
niveles de agua discretos de cada jarra.

c) Solución de 4 pasos, verificable aplicando `RESULT` a mano:

```text
(0,0) --llenar5-->    (5,0)
(5,0) --verter5→2-->  (3,2)   # la de 2 se llena, quedan 3
(3,2) --vaciar2-->    (3,0)
(3,0) --verter5→2-->  (1,2)   ✔ IS-GOAL: x = 1
```


In [ ]:
solucion = [(0, 0), (5, 0), (3, 2), (3, 0), (1, 2)]
acciones = ["llenar5", "verter5→2", "vaciar2", "verter5→2"]
print(len(acciones), "acciones; estado final:", solucion[-1])


## Solución 2 — Predicción de BFS

BFS con cola FIFO expande por niveles. Traza: se expande `A` (encola B, C),
luego `B` (encola D, E), `C` (encola F), `D` (sin hijos), `E` (encola G),
`F` (encola G de nuevo) y al desencolar `G` se detecta el objetivo.

- `expanded = [A, B, C, D, E, F, G]` (los 7 nodos, en orden de nivel).
- `path = [A, B, E, G]`: la primera copia de `G` en la cola llegó vía `E`.
- `cost = 3` (tres aristas).

El camino alternativo `A→C→F→G` también tiene costo 3, pero BFS devuelve el
que entró primero a la cola: el orden de `ACTIONS` (B antes que C) decide el empate.


In [ ]:
result = run_lab("search", seed=13)
assert result["result"]["expanded"] == ["A", "B", "C", "D", "E", "F", "G"]
assert result["result"]["path"] == ["A", "B", "E", "G"]
assert result["result"]["cost"] == 3
print("predicción verificada ✔")


## Solución 3 — Estado vs. nodo

Los dos caminos son `A→B→E→G` y `A→C→F→G`. Ambos terminan en el mismo
**estado** `G` (la misma configuración del mundo), pero generan **nodos**
distintos: cada nodo guarda su padre (`E` en un caso, `F` en el otro), la
acción que lo produjo y el costo acumulado `g`. Sin esa información extra no
se podría reconstruir la solución siguiendo punteros padre, ni detectar que
un estado ya fue alcanzado por un camino más barato.


## Solución 4 — La semilla no interviene

El grafo, el estado inicial y el objetivo están fijos en el código, y BFS es
determinista: `r13["result"] == r99["result"]` (solo difiere el campo `seed`
del contrato). Para que la semilla importara, algún componente de la
formulación debería ser aleatorio — p. ej. un grafo generado al azar o un
modelo de transición estocástico (lo que ya no sería búsqueda clásica sino un MDP).


In [ ]:
r13 = run_lab("search", seed=13)
r99 = run_lab("search", seed=99)
assert r13["result"] == r99["result"]
print("el resultado es idéntico; solo cambia la semilla registrada ✔")


## Reflexión

1. ¿Qué papel juega cada componente de la formulación (`s0`, `ACTIONS`, `RESULT`, `IS-GOAL`, `c`) en el grafo del laboratorio? Señálalos en el JSON.
2. ¿Por qué `expanded` incluye nodos como `D` y `F` que no están en `path`? ¿Qué te dice eso sobre la diferencia entre explorar y resolver?
3. El 15-puzzle tiene ≈ 10¹³ estados. ¿Por qué la formulación con `ACTIONS`/`RESULT` sigue siendo práctica aunque el grafo completo no quepa en memoria?
